# Grover attack on a toy cipher: step-by-step walkthrough

We recover the secret key **K** of the toy block cipher

$$C = S(P \oplus K)$$

from one known (plaintext, ciphertext) pair, using Grover's algorithm.

**Pipeline:** classical cipher → reversible S-box circuit → oracle → Grover iterations → simulator → (hardware).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # so `import src` works from notebooks/

import matplotlib.pyplot as plt
from src import cipher
from src.sbox_circuit import sbox_circuit
from src.oracle import build_oracle
from src.grover import (diffusion, build_grover_circuit, optimal_iterations, run_on_aer,
                        key_probabilities_per_iteration, most_likely_key, theoretical_success)
from src.metrics import circuit_metrics, get_fake_backend, compare_optimization_levels, format_table
from src.plots import mpl_key_chart, counts_to_probs

## 1. The classical cipher

Pick a block size `n` (2, 3 or 4). For `n = 4` we use the S-box from the PRESENT cipher.

In [ ]:
n = 4
sbox = cipher.get_sbox(n)
P, K = 0b1001, 0b0011           # plaintext and the SECRET key
C = cipher.encrypt(P, K, sbox)
b = lambda x: cipher.to_bits(x, n)
print("S-box:", [hex(v) for v in sbox])
print(f"P={b(P)}  K={b(K)}  ->  C={b(C)}")

key, guesses = cipher.brute_force(P, C, sbox, n)
print(f"Classical brute force found K={b(key)} after {guesses} guesses (average {(2**n+1)/2})")
print(f"Grover needs {optimal_iterations(n)} oracle calls")

## 2. The S-box as a reversible quantum circuit

A quantum circuit must be reversible, and an S-box is a permutation, so it can be built directly as
$|x\rangle \to |S(x)\rangle$ on `n` qubits. Qubit *i* holds bit *i* (Qiskit's little-endian order).

In [ ]:
s_qc = sbox_circuit(sbox, n)
print(circuit_metrics(s_qc))
s_qc.draw("mpl", fold=40)

## 3. The oracle

1. X where P has a 1 → $|K \oplus P\rangle$
2. S-box → $|S(K \oplus P)\rangle$
3. X where C has a 0 → the right key becomes $|11\ldots1\rangle$
4. Multi-controlled Z → phase flip on that state only
5. Undo 3, 2, 1

Result: $|K\rangle \to -|K\rangle$ for the secret key, every other key unchanged.

In [ ]:
oracle = build_oracle(sbox, n, P, C)
oracle.draw("mpl", fold=40)

In [ ]:
# Check: the oracle's unitary is diagonal with a single -1 at the secret key.
from qiskit.quantum_info import Operator
import numpy as np
diag = np.real(np.diag(Operator(oracle).data)).round(6)
print({b(k): int(v) for k, v in enumerate(diag)})

## 4. Diffusion and the full Grover circuit

Diffusion reflects every amplitude about the mean. Combined with the oracle's phase flip,
each iteration pushes probability onto the marked key.

In [ ]:
diffusion(n).draw("mpl")

In [ ]:
r = optimal_iterations(n)
grover = build_grover_circuit(sbox, n, P, C, iterations=r, barriers=True)
print(f"{r} iterations; ideal success probability {theoretical_success(n, r):.3f}")
grover.draw("mpl", fold=60)

## 5. Watch the key probability grow, iteration by iteration (statevector)

In [ ]:
probs = key_probabilities_per_iteration(sbox, n, P, C, iterations=r)
fig, axes = plt.subplots(1, r + 1, figsize=(4 * (r + 1), 3.2), sharey=True)
for i, ax in enumerate(axes):
    mpl_key_chart(ax, probs[i], n, K, title=f"after {i} iteration(s)")
fig.tight_layout()

## 6. Run on the Aer simulator

In [ ]:
counts = run_on_aer(build_grover_circuit(sbox, n, P, C), shots=2048, seed=7)
fig, ax = plt.subplots(figsize=(7, 3.2))
mpl_key_chart(ax, counts_to_probs(counts, n), n, K, title="Aer simulator, 2048 shots")
print("Most frequent key:", b(most_likely_key(counts)), "| secret key:", b(K))

## 7. How expensive is this on real hardware?

Real devices only run 1- and 2-qubit gates on specific qubit pairs, so the transpiler rewrites
each multi-controlled gate. We use an offline *fake* backend (no account needed).
Compare the 2-qubit gate count for n = 2, 3, 4.

In [ ]:
backend = get_fake_backend("FakeTorino")
for nn in (2, 3, 4):
    sb = cipher.get_sbox(nn)
    qc = build_grover_circuit(sb, nn, 1, cipher.encrypt(1, 2, sb))
    print(f"n = {nn}")
    print(format_table(compare_optimization_levels(qc, backend)), "\n")

## 8. Real hardware

Don't run it from here by accident. From the repo root:

```bash
python -m src.run_hardware --n 2 --plaintext 1 --key 2 --fake   # free noisy local test
python -m src.run_hardware --n 2 --plaintext 1 --key 2          # real hardware (asks to confirm)
```

Results go to `results/` and appear in the Streamlit demo (`streamlit run app.py`).